KAGGLE DA YAPILAN İŞLEMLER ÇALIŞTIRILMADAN AÇIKLAMASI İLE YAZILMIŞTIR

In [ ]:
""" VERİ YÜKLENDİ """
import polars as pl

DATA_PATH = "/kaggle/input/datasets/denizbyat/labeled/metropt3_labeled.parquet"

df = pl.read_parquet(DATA_PATH)

print(f"Satır sayısı : {df.shape[0]:,}")
print(f"Sütun sayısı : {df.shape[1]}")
print(f"Sütunlar     : {df.columns}")

In [ ]:
""" TEMİZLİK """
df = df.drop(['time_diff', 'session_id'])

# Sensör sütunları
sensor_cols = [
    'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs',
    'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric',
    'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level',
    'Caudal_impulses'
]

print(f"Temizlendi — kalan sütunlar: {df.columns}")
print(f"RAM kullanımı: {df.estimated_size('mb'):.1f} MB")

time_diff ve session_id model için gereksizdi onları temizledik

In [ ]:
""" PENCERELER VE İSTATİSTİK (Rolling Features) """
# pencerelerin boyutları
windows = { 
    '1h':  360,
    '6h':  2160,
    '24h': 8640
}

# Rolling features üret
rolling_features = [df]

for window_name, window_size in windows.items():    # pencereleri sensörlerin üzerinde gezdiriyoruz
    for col in sensor_cols:
        rolling_features.append(    # veris setine bilgileri ekliyoruz
            df.select([
                pl.col(col)
                  .rolling_mean(window_size=window_size, min_samples=1) # ortalama değerleri
                  .alias(f'{col}_rmean_{window_name}'),
                pl.col(col)
                  .rolling_std(window_size=window_size, min_samples=1) # std değerleri
                  .alias(f'{col}_rstd_{window_name}')
            ])
        )

# Hepsini birleştir
df_features = pl.concat(rolling_features, how='horizontal')

print(f"Toplam özellik sayısı : {df_features.shape[1]}")
print(f"RAM kullanımı         : {df_features.estimated_size('mb'):.1f} MB")

3 adet pencere oluşturuyoruz. Her sensöre şuan ki değer ile oluşturduğumuz pencerelerin zaman aralığında ki değerlerle normal mi. Her pencere için ortalama ve standart sapma hesaplatarak bunun değerlendirmesini yapıyoruz. Toplamda : 15 sensör × 3 pencere × 2 istatistik = 90 yeni özellik + 17 tanede var saten 107 özellik olacak

In [ ]:
""" KISA PENCERELER (Lag Features) """
lag_steps = {
    '10min': 60,   # 60 kayıt = 10 dakika
    '1h':    360,  # 360 kayıt = 1 saat
    '6h':    2160  # 2160 kayıt = 6 saat
}

lag_features = []

for lag_name, lag_size in lag_steps.items():    # lag adları ve boyutları üzerinde geziniyoruz
    for col in sensor_cols:
        lag_features.append(
            df_features.select([
                # Geçmiş değer
                pl.col(col)
                  .shift(lag_size)
                  .alias(f'{col}_lag_{lag_name}'),
                # Şimdiki - geçmiş = değişim miktarı (anlık değişim - rollingden en büyük fark)
                (pl.col(col) - pl.col(col).shift(lag_size))
                  .alias(f'{col}_diff_{lag_name}')
            ])
        )

df_features = pl.concat([df_features] + lag_features, how='horizontal')     # tüm özellikleri birleştir

print(f"Toplam özellik sayısı : {df_features.shape[1]}")
print(f"RAM kullanımı         : {df_features.estimated_size('mb'):.1f} MB")

Önceki adımda bulunan pencerelerin daha küçük ancak anlık duruma bakan halleri olarak düşünülebilir. Rolling de her pencerenin içinde kalan değerleri düzleştirerek (ortalama ve stand. sap. alarak) gürültüyü azaltıyorduk. Elimizde her pencere için tek değer kalıyor ve trend takibi yapıyorduk. Burada ise denk gelen her değeri o anki durum ile karşılaştırması yapılıyor yani ani değişimler nokta atış tespit edilerek anormallik aranıyor.

Toplamda 197 adet özellik elde edildi.